In [ ]:
library(dplyr)
library(ggplot2)
library(reshape2)

panel <- read.csv("../data/monthly_panel_vuln.csv")

In [ ]:
check_cols <- c("Binary_Artifacts_score", "Code_Review_score", "Contrib_score", 
                 "Dangerous_Workflow_score", "DependUpTool_score", "Maintained_score",
                 "Pinned_Dependencies_score", "Security_Policy_score", "Token_Permissions_score")


df_demeaned <- panel %>%
  group_by(package_name) %>%
  mutate(across(all_of(check_cols), 
                ~ .x - mean(.x, na.rm = TRUE), 
                .names = "demean_{.col}")) %>%
  ungroup()

demean_cols <- paste0("demean_", check_cols)

co_implementation_matrix <- cor(df_demeaned[, demean_cols], 
                           use = "pairwise.complete.obs")

print(round(co_implementation_matrix, 2))

In [ ]:
# Melt the matrix into a long format for ggplot
melted_matrix <- melt(co_implementation_matrix)

# Clean up variable labels
label_map <- c(
  "demean_Binary_Artifacts_score"        = "Binary-Artifacts",
  "demean_Code_Review_score"             = "Code-Review",
  "demean_Contrib_score"                 = "Contributors",
  "demean_Dangerous_Workflow_score"      = "Dangerous-Workflow",
  "demean_DependUpTool_score"            = "Dependency-Update-Tool",
  "demean_Maintained_score"              = "Maintained",
  "demean_Pinned_Dependencies_score"     = "Pinned-Dependencies",
  "demean_Security_Policy_score"         = "Security-Policy",
  "demean_Token_Permissions_score"       = "Token-Permissions"
)

clean_labels <- function(x) {
  unname(label_map[x])
}

melted_matrix$Var1 <- factor(clean_labels(as.character(melted_matrix$Var1)),
                              levels = clean_labels(levels(melted_matrix$Var1)))
melted_matrix$Var2 <- factor(clean_labels(as.character(melted_matrix$Var2)),
                              levels = clean_labels(levels(melted_matrix$Var2)))

# Plot the heatmap
ggplot(data = melted_matrix, aes(x = Var1, y = Var2, fill = value)) +
  geom_tile(color = "white") +
  scale_fill_gradient2(
    high = "#D46A85",
    #mid = "#F7F7F7",
    low = "#1A5E63",
    midpoint = 0,
    limit = c(-1, 1),
    space = "Lab",
    name = "Within-Package\nCorrelation"
  ) +
  geom_text(aes(label = round(value, 2)), color = "black", size = 3) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(),
    panel.grid.major = element_blank()
  ) +
  scale_y_discrete(limits = rev(levels(melted_matrix$Var2))) +
  coord_fixed()